# **Kaggle Challenge 2: Đề tài dự báo giá nhà.**
# **PHẦN 4: XÂY DỰNG MÔ HÌNH MÁY HỌC ĐỂ DỰ ĐOÁN GIÁ NHÀ**

## 1. Định nghĩa vấn đề
+ **Mô tả**:
   - Dự báo giá nhà dựa trên 79 tính chất của dataset nhà ở tại Ames, Iowa.
   - Dữ liệu đầu vào gồm 3 file:
      * train_processed.pkl: Dữ liệu từ tập train đã được xử lý ở phần 3.
      * test_processed.pkl: Dữ liệu từ tập  test đã được xử lý ở phần 3.
	  * train_y_processed.pkl: Dữ liệu chứa biến mục tiêu đã được xử lý ở phần 3.

+ **Mục tiêu**:
   - Dự đoán giá nhà bằng các mô hình máy học.
   - Đánh giá, cải tiến mô hình máy học. 
   - Xuất ra kết quả để submit lên Kaggle.


## 2. Chuẩn bị                

### 2.1. Import các thư viện cần thiết

In [120]:
# Load libraries
from IPython import display
import numpy as np
import pickle

import matplotlib.pyplot as plt

import pandas as pd
import seaborn as sns

# model selection
from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV, KFold

# algorithms
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor, HistGradientBoostingRegressor
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression, LinearRegression
import sklearn

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.preprocessing import RobustScaler

# metrics
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

# Hiển thị tối đa 200 cột
pd.set_option('display.max_columns', 200)

### 2.2. Đọc dataset train và test từ dataset đã xử lý trong phần dọn dẹp dữ liệu
Đọc các file .pkl trong thư mục /clean/data

In [121]:
train_data = pd.DataFrame(pd.read_pickle("../preprocessing/data/train_processed.pkl"))
test_data = pd.DataFrame(pd.read_pickle("../preprocessing/data/test_processed.pkl"))
train_y_data = pd.DataFrame(pd.read_pickle("../preprocessing/data/train_y_processed.pkl")).squeeze() # Chuyển về 1 chiều

### 2.3. Kiểm tra dữ liệu dataset train và test

In [122]:
train_data.head()                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

,MSSubClass,LotFrontage,LotArea,LotShape,LandContour,LandSlope,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,ExterQual,ExterCond,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,HeatingQC,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,MiscVal,MoSold,YrSold,MSZoning_FV,MSZoning_RH,MSZoning_RL,MSZoning_RM,Street_Pave,PavedDrive_P,PavedDrive_Y,Alley_None,Alley_Pave,LotConfig_CulDSac,LotConfig_FR2,LotConfig_FR3,LotConfig_Inside,Utilities_NoSeWa,Neighborhood_Blueste,Neighborhood_BrDale,Neighborhood_BrkSide,Neighborhood_ClearCr,Neighborhood_CollgCr,Neighborhood_Crawfor,Neighborhood_Edwards,Neighborhood_Gilbert,Neighborhood_IDOTRR,Neighborhood_MeadowV,Neighborhood_Mitchel,Neighborhood_NAmes,Neighborhood_NPkVill,Neighborhood_NWAmes,Neighborhood_NoRidge,Neighborhood_NridgHt,Neighborhood_OldTown,Neighborhood_SWISU,Neighborhood_Sawyer,Neighborhood_SawyerW,Neighborhood_Somerst,Neighborhood_StoneBr,Neighborhood_Timber,Neighborhood_Veenker,Condition1_Feedr,Condition1_Norm,Condition1_PosA,Condition1_PosN,Condition1_RRAe,Condition1_RRAn,Condition1_RRNe,Condition1_RRNn,...,Exterior1st_WdShing,Exterior2nd_AsphShn,Exterior2nd_Brk Cmn,Exterior2nd_BrkFace,Exterior2nd_CBlock,Exterior2nd_CmentBd,Exterior2nd_HdBoard,Exterior2nd_ImStucc,Exterior2nd_MetalSd,Exterior2nd_Other,Exterior2nd_Plywood,Exterior2nd_Stone,Exterior2nd_Stucco,Exterior2nd_VinylSd,Exterior2nd_Wd Sdng,Exterior2nd_Wd Shng,MasVnrType_BrkFace,MasVnrType_None,MasVnrType_Stone,Foundation_CBlock,Foundation_PConc,Foundation_Slab,Foundation_Stone,Foundation_Wood,Heating_GasA,Heating_GasW,Heating_Grav,Heating_OthW,Heating_Wall,CentralAir_Y,Electrical_FuseF,Electrical_FuseP,Electrical_Mix,Electrical_SBrkr,GarageType_Attchd,GarageType_Basment,GarageType_BuiltIn,GarageType_CarPort,GarageType_Detchd,GarageType_None,Fence_GdWo,Fence_MnPrv,Fence_MnWw,Fence_None,MiscFeature_None,MiscFeature_Othr,MiscFeature_Shed,MiscFeature_TenC,SaleType_CWD,SaleType_Con,SaleType_ConLD,SaleType_ConLI,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial,MSSubClass_30,MSSubClass_40,MSSubClass_45,MSSubClass_50,MSSubClass_60,MSSubClass_70,MSSubClass_75,MSSubClass_80,MSSubClass_85,MSSubClass_90,MSSubClass_120,MSSubClass_160,MSSubClass_180,MSSubClass_190,YrSold_2007,YrSold_2008,YrSold_2009,YrSold_2010,MoSold_sin,MoSold_cos,TotalSF,TotalRooms,TotalPorch,TotalBath,MasVnrAreaBinary,BsmtFinSF1Binary,BsmtFinSF2Binary,2ndFlrSFBinary,TotalBsmtSFBinary,LowQualFinSFBinary,WoodDeckSFBinary,OpenPorchSFBinary,EnclosedPorchBinary,3SsnPorchBinary,ScreenPorchBinary,PoolAreaBinary,MiscValBinary,GarageAreaBinary,TotalPorchBinary
0,60,-0.257516,-0.267660,3,3,2,7,5,0.949275,0.883333,-0.032711,2,2,4,3,1,6,0.183881,1,0.0,-1.183483,-0.332384,4,-0.524119,0.243325,0.0,0.342891,1,0,2,1,3,1,2,8,7,0,0,0.936364,2,2,0.293757,3,3,0.000000,-0.030571,0.000000,0.0,0.0,0.0,0,0.0,2,2008,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,8.660254e-01,5.000000e-01,-0.613178,11,-0.030571,3.5,1,1,0,1,1,0,0,1,0,0,0,0,0,1,1
1,20,0.464671,0.029682,3,3,2,6,8,0.753623,0.433333,0.000000,1,2,4,3,4,5,0.568045,1,0.0,-0.590948,0.476762,4,0.327547,0.000000,0.0,-0.327743,0,1,2,0,3,1,1,6,7,1,3,0.690

In [123]:
test_data.head()

,MSSubClass,LotFrontage,LotArea,LotShape,LandContour,LandSlope,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,ExterQual,ExterCond,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,HeatingQC,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,MiscVal,MoSold,YrSold,MSZoning_FV,MSZoning_RH,MSZoning_RL,MSZoning_RM,Street_Pave,PavedDrive_P,PavedDrive_Y,Alley_None,Alley_Pave,LotConfig_CulDSac,LotConfig_FR2,LotConfig_FR3,LotConfig_Inside,Utilities_NoSeWa,Neighborhood_Blueste,Neighborhood_BrDale,Neighborhood_BrkSide,Neighborhood_ClearCr,Neighborhood_CollgCr,Neighborhood_Crawfor,Neighborhood_Edwards,Neighborhood_Gilbert,Neighborhood_IDOTRR,Neighborhood_MeadowV,Neighborhood_Mitchel,Neighborhood_NAmes,Neighborhood_NPkVill,Neighborhood_NWAmes,Neighborhood_NoRidge,Neighborhood_NridgHt,Neighborhood_OldTown,Neighborhood_SWISU,Neighborhood_Sawyer,Neighborhood_SawyerW,Neighborhood_Somerst,Neighborhood_StoneBr,Neighborhood_Timber,Neighborhood_Veenker,Condition1_Feedr,Condition1_Norm,Condition1_PosA,Condition1_PosN,Condition1_RRAe,Condition1_RRAn,Condition1_RRNe,Condition1_RRNn,...,Exterior1st_WdShing,Exterior2nd_AsphShn,Exterior2nd_Brk Cmn,Exterior2nd_BrkFace,Exterior2nd_CBlock,Exterior2nd_CmentBd,Exterior2nd_HdBoard,Exterior2nd_ImStucc,Exterior2nd_MetalSd,Exterior2nd_Other,Exterior2nd_Plywood,Exterior2nd_Stone,Exterior2nd_Stucco,Exterior2nd_VinylSd,Exterior2nd_Wd Sdng,Exterior2nd_Wd Shng,MasVnrType_BrkFace,MasVnrType_None,MasVnrType_Stone,Foundation_CBlock,Foundation_PConc,Foundation_Slab,Foundation_Stone,Foundation_Wood,Heating_GasA,Heating_GasW,Heating_Grav,Heating_OthW,Heating_Wall,CentralAir_Y,Electrical_FuseF,Electrical_FuseP,Electrical_Mix,Electrical_SBrkr,GarageType_Attchd,GarageType_Basment,GarageType_BuiltIn,GarageType_CarPort,GarageType_Detchd,GarageType_None,Fence_GdWo,Fence_MnPrv,Fence_MnWw,Fence_None,MiscFeature_None,MiscFeature_Othr,MiscFeature_Shed,MiscFeature_TenC,SaleType_CWD,SaleType_Con,SaleType_ConLD,SaleType_ConLI,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial,MSSubClass_30,MSSubClass_40,MSSubClass_45,MSSubClass_50,MSSubClass_60,MSSubClass_70,MSSubClass_75,MSSubClass_80,MSSubClass_85,MSSubClass_90,MSSubClass_120,MSSubClass_160,MSSubClass_180,MSSubClass_190,YrSold_2007,YrSold_2008,YrSold_2009,YrSold_2010,MoSold_sin,MoSold_cos,TotalSF,TotalRooms,TotalPorch,TotalBath,MasVnrAreaBinary,BsmtFinSF1Binary,BsmtFinSF2Binary,2ndFlrSFBinary,TotalBsmtSFBinary,LowQualFinSFBinary,WoodDeckSFBinary,OpenPorchSFBinary,EnclosedPorchBinary,3SsnPorchBinary,ScreenPorchBinary,PoolAreaBinary,MiscValBinary,GarageAreaBinary,TotalPorchBinary
0,20,0.464671,0.475099,3,3,2,5,6,0.644928,0.183333,0.000000,1,2,3,3,1,3,-0.300515,2,-0.797025,-0.637935,-0.270025,2,-0.423937,0.000000,0.0,-1.083647,0,0,1,0,2,1,1,5,7,0,0,0.554545,1,1,0.972317,3,3,-0.288436,0.000000,0.0,0.0,-0.912966,0.0,0,0.000000,6,2010,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.224647e-16,-1.000000e+00,-0.693962,6,-0.912966,1.0,0,1,1,0,1,0,1,0,0,0,1,0,0,1,1
1,20,0.507940,0.952946,2,3,2,6,6,0.623188,0.133333,-0.587195,1,2,3,3,1,5,0.499805,1,0.000000,-0.258563,0.584609,2,0.441062,0.000000,0.0,-0.213560,0,0,1,1,3,

In [124]:
train_y_data.head()

0    0.493727
1    0.215600
2    0.633052
3   -0.305049
4    0.857766
Name: SalePrice, dtype: float64

In [125]:
print("Số cột null trong train: ", train_data.isnull().sum().gt(0).sum())
print("Số cột null trong test: ", test_data.isnull().sum().gt(0).sum())

Số cột null trong train:  0
Số cột null trong test:  0


In [126]:
train_data.dtypes.value_counts()

float64    195
int64       48
Name: count, dtype: int64

In [127]:
test_data.dtypes.value_counts()

float64    195
int64       48
Name: count, dtype: int64

## 3. Thiết lập các tham số mặc định

In [128]:
parameters = {}
parameters["random_state"] = 42
parameters["k_fold"] = 10

## 4. Huấn luyện mô hình

### 4.1. Chia K-Fold 
**K-Fold Cross Validation** là kỹ thuật đánh giá mô hình học máy bằng cách chia dữ liệu thành **K phần (fold)** có kích thước gần bằng nhau.  
Quá trình huấn luyện và kiểm tra sẽ được lặp lại **K lần**, mỗi lần sử dụng một fold khác nhau làm **tập kiểm tra (validation)**, và **K-1 fold còn lại** làm **tập huấn luyện (training)**.


In [129]:
kf = KFold(n_splits=parameters["k_fold"], shuffle=True, random_state=parameters["random_state"])

### 4.2. Chọn các model để train.
#### Logistic Regression
**Linear Regression** là mô hình hồi quy cơ bản và phổ biến nhất, được sử dụng để **dự đoán giá trị liên tục** dựa trên mối quan hệ **tuyến tính** giữa các biến đầu vào (độc lập) và biến đầu ra (phụ thuộc).

- Mục tiêu của mô hình là **tìm đường thẳng (hoặc siêu phẳng trong không gian nhiều chiều)** sao cho sai số giữa giá trị dự đoán và giá trị thực tế là nhỏ nhất.

#### Random Forest
**Random Forest** là mô hình **ensemble learning** (học tập hợp) dựa trên nhiều **cây quyết định (Decision Tree)**.  
Kết quả cuối cùng là **trung bình** (hồi quy) hoặc **bỏ phiếu đa số** (phân loại) từ nhiều cây con.

- Tăng cường độ chính xác và giảm overfitting nhờ cơ chế **bagging** (bootstrap aggregation).  
- Mỗi cây được huấn luyện trên một **mẫu dữ liệu ngẫu nhiên** và chỉ dùng một phần các đặc trưng.  

#### XGBoost
**XGBoost (Extreme Gradient Boosting)** là một mô hình **boosting** mạnh mẽ, xây dựng cây liên tiếp để **sửa lỗi** của các cây trước.

- Mỗi cây mới được huấn luyện dựa trên **gradient của hàm mất mát**, giúp giảm lỗi nhanh.  
- Hỗ trợ regularization (L1, L2) → giảm overfitting.  
- Tối ưu hiệu suất với **song song hóa** và **xử lý thiếu dữ liệu tự động**.  


#### CatBoost
**CatBoost (Categorical Boosting)** là thuật toán boosting do Yandex phát triển, đặc biệt tối ưu cho **biến phân loại (categorical features)**.

- Tự động **mã hóa biến phân loại** bằng cơ chế **target statistics** an toàn (không gây rò rỉ dữ liệu).  
- Dựa trên **Ordered Boosting** giúp giảm overfitting so với gradient boosting thông thường.  
- Là mô hình mạnh, **ít cần tiền xử lý dữ liệu** và **dễ dùng**.  

#### Support Vector Machine (SVM)
**SVM** tìm **siêu phẳng (hyperplane)** tối ưu để phân tách các lớp dữ liệu sao cho **khoảng cách biên (margin)** giữa các lớp là lớn nhất.

- Dữ liệu không tuyến tính có thể được ánh xạ sang không gian cao hơn bằng **kernel trick** (RBF, polynomial,...). 

#### Light GBM
**LightGBM** (Light Gradient Boosting Machine) được phát triển bởi **Microsoft Research**, tối ưu hóa từ ý tưởng của XGBoost.
- Là thuật toán boosting cây quyết định theo gradient (Gradient Boosting Decision Tree - GBDT)
- Giữ độ chính xác tương đương hoặc tốt hơn XGBoost, nhưng tốc độ nhanh hơn nhiều lần nhờ các cải tiến cốt lõi.



In [130]:
models = {}

# Random Forest Regressor
models["Random Forest"] = RandomForestRegressor(
    random_state=parameters["random_state"],
    n_jobs=-1,
    max_depth=8,             # tăng độ sâu một chút cho dữ liệu phức tạp
    min_samples_leaf=2,
    n_estimators=500,         # nhiều cây hơn thường cho kết quả ổn định hơn
)

# XGBoost Regressor
models["XGBoost"] = XGBRegressor(
    max_depth=8,
    learning_rate=0.05,
    n_estimators=500,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=parameters["random_state"],
    n_jobs=-1,
    eval_metric='rmse'        # metric chuẩn cho hồi quy
)

# Support Vector Regressor (SVR)
models["SVM"] = SVR(
    kernel='rbf',
    C=50000,
    epsilon=0.1               # biên dung sai cho hồi quy epsilon-SVR
)

# CatBoost Regressor
models["CatBoost"] = CatBoostRegressor(
    depth=8,
    l2_leaf_reg=3,
    subsample=0.8,
    learning_rate=0.05,
    iterations=1000,
    random_seed=parameters["random_state"],
    loss_function='RMSE',     # đúng loss cho hồi quy
    eval_metric='RMSE',
    verbose=0,
    allow_writing_files=False
)

# LightGBM
models["LightGBM"] = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=3,
    random_state=parameters["random_state"],
    n_jobs=-1,
)


### 4.3. Thiết lập Stacking Model
Dùng StackingRegressor để kết hợp dự đoán từ nhiều mô hình học máy khác nhau nhằm đạt được hiệu suất dự đoán tốt hơn.

In [131]:
meta_model = LinearRegression(n_jobs=-1)
base_learners = [(name, models[name]) for name in models.keys()] 

stacked_model = StackingRegressor(
    estimators=base_learners,
    final_estimator=meta_model,
    cv=parameters["k_fold"],
    n_jobs=-1,
	verbose=2
)


## 5. Thực hiện việc tái thiết lập biến mục tiêu

In [132]:
y_data_unprocessed = pd.DataFrame(pd.read_pickle("../preprocessing/data/train_y_unprocessed.pkl"))

In [133]:
y_data_unprocessed.head()

,SalePrice
0,208500
1,181500
2,223500
3,140000
4,250000


In [134]:
# Tái tiền xử lý dữ liệu để thiết lập lại scaler, log transformation.
scaler = RobustScaler()
y_data_unprocessed = np.log1p(y_data_unprocessed)
y_data_unprocessed = scaler.fit_transform(y_data_unprocessed.values.reshape(-1, 1))

## 6. Chọn mô hình tốt nhất để dự đoán giá nhà trên tập test và xuất kết quả ra file csv

In [135]:
stacked_model.fit(train_data, train_y_data)

y_test_pred = stacked_model.predict(test_data)

# Nghịch đảo lại scaler, log transformation (do biến này đã được biến đổi trong tiền xử lý dữ liệu), trả về dữ liệu thật
y_pred_real = scaler.inverse_transform(y_test_pred.reshape(-1, 1)) # Đảo ngược RobustScaler
y_pred_real = np.expm1(y_pred_real) # Đảo ngược log transformation

# Chuyển thành vector 1 chiều
y_pred_real = y_pred_real.ravel()  

submission = pd.DataFrame({
    "Id": range(1461, len(y_pred_real) + 1461),
    "SalePrice": y_pred_real
})

submission.to_csv("submission.csv", index=False)
print("Đã tạo ra file submission.csv")


Đã tạo ra file submission.csv


## 7. Xuất các file backup
Lưu thành file html và file ipynb trong thư mục backup.

In [136]:
from datetime import datetime
import os

# Lấy ngày và giờ
timestamp = datetime.now().strftime("%d-%m_%H-%M")

# Thêm ngày giờ vào notebook
notebook_name = f"model_{timestamp}"

# Chọn thư mục lưu vào
save_dir = "..\\exps\\model"

# Xuất ra file html và ipynb.
os.system(f"jupyter nbconvert --to html --output {save_dir}\\{notebook_name}.html model.ipynb")
os.system(f"copy model.ipynb {save_dir}\\{notebook_name}.ipynb")

0

# **Kết thúc**